# Detecting expr4.frkern table (0602)

Run this notebook on af309 with the `xfold-scripts` kernel. Set `RUN_ROOTS` to one or more `detecting.expr4.frkern/` data directories. Set `CDF_KIND = 'ta'` or `'tb'` to choose the measured CDF and corresponding theoretical walking-list values. This notebook does not draw figures; it computes `Rl` and `Rh` tables indexed by front kernel and rear kernel, then emits LaTeX table text.


In [2]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd

# Set one or more host data directories. Each root should point to detecting.expr4.frkern/.
CDF_KIND = 'ta'  # choose 'ta' or 'tb'

RUN_ROOTS = [
    Path('/astrum/home/hpchzy/code/data/20260603/c920bn3/outputDetecting/expr4frkern/walk20/20260603-191949/detecting.expr4.frkern/'),
    Path('/astrum/home/hpchzy/code/data/20260602/c920bn3/outputDetecting/expr4frkern/walk20/20260602-192033/detecting.expr4.frkern/'),
    # Path('/astrum/home/hpchzy/code/data/20260602/camd9554n2/outputDetecting/expr4frkern/walk20/20260602-full0602/detecting.expr4.frkern/'),
]

DROP_TIMERS = {'cntvct_fence', 'tsc_asym'}
RL_QUANTILE_BOUNDARY = (0.0, 0.9)
RH_QUANTILE_BOUNDARY = (0.9, 0.99)

HOST_LABELS = {
    'c920bn3': 'Kunpeng 920B',
    'camd9554n2': 'AMD EPYC 9554',
    'cgnr6760pn2': 'Intel Xeon 6760P',
}

KERNEL_ORDER = ['none', 'copy', 'scale', 'add', 'triad', 'jacobi2d5p', 'tl_f90_cg_calc_w']

def kernel_sort_key(name):
    name = str(name)
    return (KERNEL_ORDER.index(name) if name in KERNEL_ORDER else len(KERNEL_ORDER), name)

if CDF_KIND not in {'ta', 'tb'}:
    raise ValueError("CDF_KIND must be 'ta' or 'tb'")
print(f'CDF_KIND={CDF_KIND}')
for root in RUN_ROOTS:
    print(root)


CDF_KIND=ta
/astrum/home/hpchzy/code/data/20260603/c920bn3/outputDetecting/expr4frkern/walk20/20260603-191949/detecting.expr4.frkern
/astrum/home/hpchzy/code/data/20260602/c920bn3/outputDetecting/expr4frkern/walk20/20260602-192033/detecting.expr4.frkern


In [3]:
from scipy.stats import wasserstein_distance
def parse_meta(path):
    d = {}
    if not path.exists():
        return d
    for line in path.read_text(errors='ignore').splitlines():
        if '=' in line:
            k, v = line.split('=', 1)
            d[k.strip()] = v.strip()
    return d


def read_values(path):
    vals = []
    for line in path.read_text(errors='ignore').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        try:
            vals.append(float(line.split(',')[-1]))
        except ValueError:
            pass
    return vals


def q(vals, p):
    vals = sorted(float(v) for v in vals if math.isfinite(float(v)))
    if not vals:
        return math.nan
    if len(vals) == 1:
        return vals[0]
    pos = p * (len(vals) - 1)
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return vals[lo]
    return vals[lo] * (hi - pos) + vals[hi] * (pos - lo)


def ta_from_run_dir(run_dir, meta):
    for key in ('ta', 'mu_ns', 'interval_ns'):
        if key in meta:
            try:
                return float(meta[key])
            except ValueError:
                pass
    m = re.search(r'_ta([0-9]+(?:\.[0-9]+)?)', run_dir.name)
    if m:
        return float(m.group(1))
    return math.nan

def tb_from_run_dir(run_dir, meta):
    for key in ('tb', 'mu_ns', 'interval_ns'):
        if key in meta:
            try:
                return float(meta[key])
            except ValueError:
                pass
    m = re.search(r'_tb([0-9]+(?:\.[0-9]+)?)', run_dir.name)
    if m:
        return float(m.group(1))
    return math.nan


def finite_array(vals):
    arr = np.asarray(vals, dtype=float)
    return arr[np.isfinite(arr)]


def quantile_slice(arr, lo, hi):
    arr = finite_array(arr)
    if arr.size == 0:
        return arr
    qlo = np.quantile(arr, lo)
    qhi = np.quantile(arr, hi)
    return arr[(arr >= qlo) & (arr <= qhi)]


def calc_rl_rh(measured, theory):
    measured = finite_array(measured)
    theory = finite_array(theory)
    if measured.size == 0 or theory.size == 0:
        return {k: math.nan for k in ['rl', 'rh', 'rl_numerator', 'rl_denominator', 'rh_numerator', 'rh_denominator']}

    measured_aligned = measured - np.min(measured)
    theory_aligned = theory - np.min(theory)

    measured_rl = quantile_slice(measured_aligned, *RL_QUANTILE_BOUNDARY)
    theory_rl = quantile_slice(theory_aligned, *RL_QUANTILE_BOUNDARY)
    measured_rh = quantile_slice(measured_aligned, *RH_QUANTILE_BOUNDARY)
    theory_rh = quantile_slice(theory_aligned, *RH_QUANTILE_BOUNDARY)

    rl_numerator = wasserstein_distance(measured_rl, theory_rl)
    rl_denominator = wasserstein_distance(measured_rl, np.zeros_like(measured_rl))
    rh_numerator = wasserstein_distance(measured_rh, theory_rh)
    rh_denominator = wasserstein_distance(measured_rh, np.zeros_like(measured_rh))

    return {
        'rl': rl_numerator / rl_denominator if rl_denominator else math.nan,
        'rh': rh_numerator / rh_denominator if rh_denominator else math.nan,
        'rl_numerator': rl_numerator,
        'rl_denominator': rl_denominator,
        'rh_numerator': rh_numerator,
        'rh_denominator': rh_denominator,
    }



def target_from_run_dir(run_dir, meta, kind):
    if kind == 'ta':
        return ta_from_run_dir(run_dir, meta)
    if kind == 'tb':
        return tb_from_run_dir(run_dir, meta)
    raise ValueError(f'unknown CDF kind: {kind}')


In [4]:
rows = []
for run_root in RUN_ROOTS:
    for cdf in sorted(run_root.rglob(f'*_{CDF_KIND}_cdf.csv')):
        run_dir = cdf.parent
        combo_dir = run_dir.parent
        meta = parse_meta(run_dir / 'meta.txt') or parse_meta(combo_dir / 'meta.txt')
        measured = read_values(cdf)
        if not measured:
            continue
        ta = ta_from_run_dir(run_dir, meta)
        tb = tb_from_run_dir(run_dir, meta)
        target = target_from_run_dir(run_dir, meta, CDF_KIND)
        rows.append({
            'run_root': str(run_root),
            'host': meta.get('host', run_root.parts[-6] if len(run_root.parts) >= 6 else ''),
            'expr': meta.get('expr_name', run_root.name),
            'np': int(float(meta.get('np', 0) or 0)),
            'timer': meta.get('timer', cdf.parents[2].name),
            'interval_ns': int(float(meta.get('interval_ns', meta.get('mu_ns', 0)) or 0)),
            'fsize_kib': int(float(meta.get('fsize_kib', 0) or 0)),
            'fkern': meta.get('fkern', ''),
            'rkern': meta.get('rkern', ''),
            'ta': ta,
            'tb': tb,
            'n_measured': len(measured),
            'measured': measured,
            'cdf_kind': CDF_KIND,
            'target': target,
            'theory': [target] * len(measured),
            'run_dir': str(run_dir),
        })

raw = pd.DataFrame(rows)
print(f'roots: {len(RUN_ROOTS)}, cdf files: {len(raw)}')
if raw.empty:
    raise RuntimeError(f'No *_{CDF_KIND}_cdf.csv files found under RUN_ROOTS')
raw = raw[~raw['timer'].isin(DROP_TIMERS)].copy()
raw.head()


roots: 2, cdf files: 5040


,run_root,host,expr,np,timer,interval_ns,fsize_kib,fkern,rkern,ta,tb,n_measured,measured,cdf_kind,target,theory,run_dir
0,/astrum/home/hpchzy/code/data/20260603/c920bn3...,c920bn3,detecting.expr4.frkern,64,clock_gettime,1000,0,add,add,1000.0,1000.0,100,"[1030.0, 1030.0, 1030.0, 1030.0, 1030.0, 1030....",ta,1000.0,"[1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000....",/astrum/home/hpchzy/code/data/20260603/c920bn3...
1,/astrum/home/hpchzy/code/data/20260603/c920bn3...,c920bn3,detecting.expr4.frkern,64,clock_gettime,1000,0,add,add,998.0,998.0,100,"[1020.0, 1020.0, 1030.0, 1030.0, 1030.0, 1030....",ta,998.0,"[998.0, 998.0, 998.0, 998.0, 998.0, 998.0, 998...",/astrum/home/hpchzy/code/data/20260603/c920bn3...
2,/astrum/home/hpchzy/code/data/20260603/c920bn3...,c920bn3,detecting.expr4.frkern,64,clock_gettime,1000,0,add,add,1001.0,1001.0,100,"[1030.0, 1030.0, 1030.0, 1030.0, 1030.0, 1030....",ta,1001.0,"[1001.0, 1001.0, 1001.0, 1001.0, 1001.0, 1001....",/astrum/home/hpchzy/code/data/20260603/c920bn3...
3,/astrum/home/hpchzy/code/data/20260603/c920bn3...,c920bn3,detecting.expr4.frkern,64,clock_gettime,1000,0,add,add,1001.0,1001.0,100,"[1030.0, 1030.0, 1030.0, 1030.0, 1030.0, 1030....",ta,1001.0,"[1001.0, 1001.0, 1001.0, 1001.0, 1001.0, 1001....",/astrum/home/hpchzy/code/data/20260603/c920bn3...
4,/astrum/home/hpchzy/code/data/20260603/c920bn3...,c920bn3,detecting.expr4.frkern,64,clock_gettime,1000,0,add,add,997.0,997.0,100,"[1020.0, 1020.0, 1020.0, 1020.0, 1020.0, 1030....",ta,997.0,"[997.0, 997.0, 997.0, 997.0, 997.0, 997.0, 997...",/astrum/home/hpchzy/code/data/20260603/c920bn3...


In [5]:
summary_rows = []
keys = ['host', 'expr', 'np', 'timer', 'interval_ns', 'fsize_kib', 'fkern', 'rkern']
for key, g in raw.groupby(keys, dropna=False):
    measured = [x for vals in g['measured'] for x in vals]
    theory = [x for vals in g['theory'] for x in vals]
    metrics = calc_rl_rh(measured, theory)
    row = dict(zip(keys, key))
    row.update(metrics)
    row.update({
        'cdf_kind': CDF_KIND,
        'n_cdf_files': len(g),
        'n_measured': sum(len(vals) for vals in g['measured']),
        'ta_unique': g['ta'].nunique(),
        'tb_unique': g['tb'].nunique(),
        'target_unique': g['target'].nunique(),
        'measured_mean_ns': float(np.mean(measured)) if measured else math.nan,
        'measured_p99_ns': float(np.quantile(measured, 0.99)) if measured else math.nan,

        "measured_q5_ns": float(np.quantile(measured, 0.05)) if measured else math.nan,
        "measured_q25_ns": float(np.quantile(measured, 0.25)) if measured else math.nan,
        "measured_q50_ns": float(np.quantile(measured, 0.5)) if measured else math.nan,
        "measured_q75_ns": float(np.quantile(measured, 0.75)) if measured else math.nan,
        "measured_q95_ns": float(np.quantile(measured, 0.95)) if measured else math.nan,

        "target_q5_ns": float(np.quantile(theory, 0.05)) if theory else math.nan,
        "target_q25_ns": float(np.quantile(theory, 0.25)) if theory else math.nan,
        "target_q50_ns": float(np.quantile(theory, 0.5)) if theory else math.nan,
        "target_q75_ns": float(np.quantile(theory, 0.75)) if theory else math.nan,
        "target_q95_ns": float(np.quantile(theory, 0.95)) if theory else math.nan,

    })

    summary_rows.append(row)

df = pd.DataFrame(summary_rows).sort_values(['host', 'timer', 'interval_ns', 'fkern', 'rkern'])
df['host_label'] = df['host'].map(HOST_LABELS).fillna(df['host'])
print(f'summary rows: {len(df)}')
df.head()
df.to_csv(f"demo.expr4Frkern.0603.csv", index=False)


summary rows: 216


In [6]:
# Build Rl/Rh front-kernel x rear-kernel tables. No figures are generated.
tables = {}
for (host, timer, interval_ns, fsize_kib), g in df.groupby(['host', 'timer', 'interval_ns', 'fsize_kib'], dropna=False):
    fk_order = sorted(g['fkern'].dropna().unique(), key=kernel_sort_key)
    rk_order = sorted(g['rkern'].dropna().unique(), key=kernel_sort_key)
    rl_table = g.pivot_table(index='fkern', columns='rkern', values='rl', aggfunc='mean').reindex(index=fk_order, columns=rk_order)
    rh_table = g.pivot_table(index='fkern', columns='rkern', values='rh', aggfunc='mean').reindex(index=fk_order, columns=rk_order)
    tables[(host, timer, int(interval_ns), int(fsize_kib))] = {'Rl': rl_table, 'Rh': rh_table}
    print(f'== {HOST_LABELS.get(host, host)} timer={timer} interval={int(interval_ns)}ns fsize={int(fsize_kib)}KiB ==')
    print('Rl')
    display(rl_table.round(4))
    print('Rh')
    display(rh_table.round(4))

print(f'tables: {len(tables)}')


== Kunpeng 920B timer=clock_gettime interval=1000ns fsize=0KiB ==
Rl


rkern,copy,scale,add,triad,dgemm,pow
fkern,,,,,,
copy,0.8150,0.8149,0.8146,0.8149,0.8148,0.8147
scale,0.8148,0.8151,0.8154,0.8149,0.8157,0.8150
add,0.8152,0.8144,0.8150,0.8156,0.8153,0.8150
triad,0.8154,0.8152,0.8146,0.8153,0.8153,0.8149
dgemm,0.8148,0.8149,0.8152,0.8149,0.8154,0.8145
pow,0.8151,0.8151,0.8154,0.8146,0.8152,0.8148


Rh


rkern,copy,scale,add,triad,dgemm,pow
fkern,,,,,,
copy,0.8022,0.8028,0.8026,0.8030,0.8028,0.8027
scale,0.8030,0.8023,0.8026,0.8026,0.8023,0.8024
add,0.8020,0.8025,0.8022,0.8026,0.8030,0.8025
triad,0.8027,0.8027,0.8020,0.8051,0.8047,0.8023
dgemm,0.8029,0.8027,0.8025,0.8026,0.8024,0.8024
pow,0.8032,0.8031,0.8033,0.8027,0.8028,0.8026


== Kunpeng 920B timer=cntvct interval=1000ns fsize=0KiB ==
Rl


...

== Kunpeng 920B timer=papix6 interval=1000ns fsize=0KiB ==
Rl


rkern,copy,scale,add,triad,dgemm,pow
fkern,,,,,,
copy,0.8250,0.8236,0.8248,0.8241,0.8251,0.8252
scale,0.8264,0.8241,0.8261,0.8256,0.8259,0.8262
add,0.8284,0.8267,0.8285,0.8300,0.8289,0.8297
triad,0.8252,0.8233,0.8259,0.8251,0.8248,0.8250
dgemm,0.8254,0.8238,0.8256,0.8253,0.8239,0.8237
pow,0.8248,0.8245,0.8252,0.8252,0.8259,0.8246


Rh


rkern,copy,scale,add,triad,dgemm,pow
fkern,,,,,,
copy,0.9011,0.8995,0.9022,0.8984,0.9029,0.8941
scale,0.8988,0.8991,0.8997,0.9016,0.8977,0.8990
add,0.8974,0.8943,0.8952,0.8964,0.9006,0.9021
triad,0.8954,0.9014,0.8965,0.8981,0.8978,0.8987
dgemm,0.9027,0.8917,0.8932,0.9059,0.9042,0.9038
pow,0.9028,0.8962,0.9009,0.9025,0.8993,0.9009


tables: 6


In [7]:
# Emit one LaTeX table* per selected (host, timer, interval, fsize).
# The table layout matches the paper style: rows are Rear Flush kernels,
# columns are Front Flush kernels, each with R_L and R_H subcolumns.
SELECT_HOST = None          # e.g., 'c920bn3'
SELECT_TIMER = 'cntvct'    # e.g., 'cntvcto'; set None to emit all timers
SELECT_INTERVAL_NS = None
SELECT_FSIZE_KIB = None
ROUND_DIGITS = 1
KERNEL_TABLE_ORDER = ['copy', 'add', 'scale', 'triad', 'pow', 'dgemm']


def latex_escape(s):
    return str(s).replace('_', r'\_')


def kernel_label(name):
    return latex_escape(str(name).upper())


def table_label(host, timer):
    return f"tab:expr-frkern-{host}-{timer}-{CDF_KIND}".replace('_', '-')


def combined_rl_rh_latex(host, timer, interval_ns, fsize_kib, rl_table, rh_table):
    front_order = [k for k in KERNEL_TABLE_ORDER if k in rl_table.columns or k in rh_table.columns]
    rear_order = [k for k in KERNEL_TABLE_ORDER if k in rl_table.index or k in rh_table.index]
    if not front_order:
        front_order = list(rl_table.columns)
    if not rear_order:
        rear_order = list(rl_table.index)

    host_name = HOST_LABELS.get(host, host)
    col_spec = '@{\\extracolsep{\\fill}}l' + 'cc' * len(front_order)
    ncols = 1 + 2 * len(front_order)
    lines = []
    lines.append(r'\begin{table*}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\caption{' + f'{host_name} {latex_escape(timer)} ({CDF_KIND}): $R_L$ and $R_H$ with different flush kernels (\\%)' + r'}')
    lines.append(r'\label{' + table_label(host, timer) + r'}')
    lines.append(r'\begin{tabular*}{\textwidth}{' + col_spec + r'}')
    lines.append(r'\toprule')
    lines.append(r'\multirow{3}{*}{Rear Flush} & \multicolumn{' + str(2 * len(front_order)) + r'}{c}{Front Flush} \\ \cmidrule(lr){2-' + str(ncols) + r'}')
    group_cells = []
    cmidrules = []
    for idx, front in enumerate(front_order):
        start = 2 + 2 * idx
        end = start + 1
        group_cells.append(r'\multicolumn{2}{c}{' + kernel_label(front) + r'}')
        cmidrules.append(r'\cmidrule(lr){' + f'{start}-{end}' + r'}')
    lines.append('& ' + ' & '.join(group_cells) + r' \\ ' + ' '.join(cmidrules))
    metric_cells = []
    for _front in front_order:
        metric_cells.extend([r'$R_L$', r'$R_H$'])
    lines.append('& ' + ' & '.join(metric_cells) + r' \\ \midrule')

    for rear in rear_order:
        row = [kernel_label(rear)]
        for front in front_order:
            rl = rl_table.loc[rear, front] if rear in rl_table.index and front in rl_table.columns else float('nan')
            rh = rh_table.loc[rear, front] if rear in rh_table.index and front in rh_table.columns else float('nan')
            row.append('--' if pd.isna(rl) else f'{100 * rl:.{ROUND_DIGITS}f}')
            row.append('--' if pd.isna(rh) else f'{100 * rh:.{ROUND_DIGITS}f}')
        lines.append(' & '.join(row) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular*}')
    lines.append(r'\end{table*}')
    return '\n'.join(lines)

latex_outputs = []
for (host, timer, interval_ns, fsize_kib), pair in tables.items():
    if SELECT_HOST is not None and host != SELECT_HOST:
        continue
    if SELECT_TIMER is not None and timer != SELECT_TIMER:
        continue
    if SELECT_INTERVAL_NS is not None and interval_ns != SELECT_INTERVAL_NS:
        continue
    if SELECT_FSIZE_KIB is not None and fsize_kib != SELECT_FSIZE_KIB:
        continue
    latex_outputs.append(combined_rl_rh_latex(host, timer, interval_ns, fsize_kib, pair['Rl'], pair['Rh']))

print('\n\n'.join(latex_outputs))


\begin{table*}[htbp]
\centering
\caption{Kunpeng 920B cntvct (ta): $R_L$ and $R_H$ with different flush kernels (\%)}
\label{tab:expr-frkern-c920bn3-cntvct-ta}
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lcccccccccccc}
\toprule
\multirow{3}{*}{Rear Flush} & \multicolumn{12}{c}{Front Flush} \\ \cmidrule(lr){2-13}
& \multicolumn{2}{c}{COPY} & \multicolumn{2}{c}{ADD} & \multicolumn{2}{c}{SCALE} & \multicolumn{2}{c}{TRIAD} & \multicolumn{2}{c}{POW} & \multicolumn{2}{c}{DGEMM} \\ \cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9} \cmidrule(lr){10-11} \cmidrule(lr){12-13}
& $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ \\ \midrule
COPY & 76.5 & 79.4 & 76.6 & 79.4 & 76.5 & 79.4 & 76.6 & 79.4 & 76.5 & 79.4 & 76.6 & 79.4 \\
ADD & 76.6 & 79.4 & 76.6 & 79.7 & 76.6 & 79.4 & 76.6 & 79.8 & 76.5 & 79.5 & 76.5 & 79.4 \\
SCALE & 76.6 & 79.4 & 76.6 & 79.4 & 76.6 & 79.4 & 76.6 & 79.4 & 76.6 & 79.4 & 76.6 & 79.4 \\
TRIAD & 76.